# PPO — Proximal Policy Optimization Algorithms (2017)

_Papers · Reinforcement Learning_

**Clip the policy-update ratio inside the objective, so you can take big, reusable gradient steps without the policy collapsing.**

---

This notebook is a practice scaffold for the **PPO — Proximal Policy Optimization Algorithms (2017)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q gymnasium
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — gymnasium + PyTorch (Colab)

In [ ]:
# In Colab first run:  !pip install gymnasium
# (torch is preinstalled in Colab; gymnasium is not.)
import torch
import torch.nn as nn
from torch.distributions import Categorical
import gymnasium as gym

torch.manual_seed(0)

# --- 0. Sanity-check the lesson's worked example: r = 0.8/0.5, A = +2, eps = 0.2. ---
EPS = 0.2
r   = torch.tensor(0.8 / 0.5)              # probability ratio = 1.6
A   = torch.tensor(2.0)                     # positive advantage
unclipped = r * A                          # 1.6 * 2 = 3.2
clipped   = torch.clamp(r, 1 - EPS, 1 + EPS) * A   # clip 1.6 -> 1.2, * 2 = 2.4
print("worked example:  r =", r.item(), " unclipped =", unclipped.item(),
      " clipped =", clipped.item(), " L_CLIP = min =", torch.min(unclipped, clipped).item())
# worked example:  r = 1.6  unclipped = 3.2  clipped = 2.4  L_CLIP = min = 2.4


# --- 1. Policy + value networks (Track B: nn.Linear primitives). ---
class ActorCritic(nn.Module):
    def __init__(self, obs_dim, n_act, hidden=64):
        super().__init__()
        self.body   = nn.Sequential(nn.Linear(obs_dim, hidden), nn.Tanh(),
                                    nn.Linear(hidden, hidden), nn.Tanh())
        self.pi     = nn.Linear(hidden, n_act)   # action logits  -> policy
        self.v      = nn.Linear(hidden, 1)        # state value V(s) -> critic

    def forward(self, x):
        h = self.body(x)
        return Categorical(logits=self.pi(h)), self.v(h).squeeze(-1)


# --- 2. GAE advantage (Eqs. 11-12): delta_t = r + gamma V' - V; A_t = sum (gamma lam)^l delta. ---
def compute_gae(rewards, values, dones, last_v, gamma=0.99, lam=0.95):
    adv = torch.zeros(len(rewards)); gae = 0.0
    values = values + [last_v]
    for t in reversed(range(len(rewards))):
        mask  = 1.0 - dones[t]                         # 0 if episode ended at t
        delta = rewards[t] + gamma * values[t + 1] * mask - values[t]   # Eq. 12
        gae   = delta + gamma * lam * mask * gae       # Eq. 11 (recursive form)
        adv[t] = gae
    returns = adv + torch.tensor(values[:-1])          # value targets for the critic
    return adv, returns


# --- 3. One PPO update on a collected batch: the clipped loss (Eq. 7) + value + entropy (Eq. 9). ---
def ppo_update(net, opt, obs, acts, old_logp, adv, returns,
               clip_eps=0.2, c1=0.5, c2=0.01, epochs=10, use_clip=True):
    adv = (adv - adv.mean()) / (adv.std() + 1e-8)      # normalize advantages
    for _ in range(epochs):                            # reuse the batch -- safe via the clip
        dist, value = net(obs)
        new_logp = dist.log_prob(acts)
        ratio    = torch.exp(new_logp - old_logp)      # r_t = pi_theta / pi_theta_old
        if use_clip:
            unc = ratio * adv
            clp = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * adv
            policy_loss = -torch.min(unc, clp).mean()  # Eq. 7:  -L^CLIP
        else:
            policy_loss = -(ratio * adv).mean()         # ABLATION: raw surrogate, no clip
        value_loss = (returns - value).pow(2).mean()    # L^VF
        entropy    = dist.entropy().mean()              # S
        loss = policy_loss + c1 * value_loss - c2 * entropy   # Eq. 9
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(net.parameters(), 0.5)  # gradient clip (separate from L^CLIP)
        opt.step()


# --- 4. Train until CartPole is solved; PRINT the return rising. ---
def train(use_clip=True, updates=60, steps_per=2048):
    torch.manual_seed(0)
    env = gym.make("CartPole-v1")
    net = ActorCritic(env.observation_space.shape[0], env.action_space.n)
    opt = torch.optim.Adam(net.parameters(), lr=3e-4)
    obs, _ = env.reset(seed=0)
    ep_ret, recent = 0.0, []
    history = []
    for up in range(updates):
        O, Ac, LP, R, V, D = [], [], [], [], [], []
        for _ in range(steps_per):                       # collect a rollout (ON-POLICY)
            ot = torch.as_tensor(obs, dtype=torch.float32)
            with torch.no_grad():
                dist, v = net(ot)
                a = dist.sample()
                lp = dist.log_prob(a)                     # log pi_old recorded at collection
            nobs, rew, term, trunc, _ = env.step(int(a))
            done = term or trunc
            O.append(ot); Ac.append(a); LP.append(lp); R.append(float(rew))
            V.append(float(v)); D.append(1.0 if done else 0.0)
            ep_ret += rew; obs = nobs
            if done:
                recent.append(ep_ret); ep_ret = 0.0
                obs, _ = env.reset()
        with torch.no_grad():
            last_v = float(net(torch.as_tensor(obs, dtype=torch.float32))[1])
        adv, ret = compute_gae(R, V, D, last_v)
        ppo_update(net, opt, torch.stack(O), torch.stack(Ac), torch.stack(LP),
                   adv, ret, epochs=10, use_clip=use_clip)
        avg = sum(recent[-20:]) / max(1, len(recent[-20:]))
        history.append(avg)
        print(f"  update {up:2d}  avg return (last 20 eps): {avg:6.1f}")
        recent = recent[-20:]
        if avg >= 475:                                   # CartPole-v1 is "solved" at 475+
            print("  -> SOLVED CartPole."); break
    env.close()
    return history

print("PPO with clip (Eq. 7):")
clip_hist = train(use_clip=True)
print("\nABLATION -- no clip (raw r_t*A_t, same everything else):")
noclip_hist = train(use_clip=False)
print("\nClipped   avg-return trajectory:", [round(h, 1) for h in clip_hist])
print("No-clip   avg-return trajectory:", [round(h, 1) for h in noclip_hist])
# Clipped PPO climbs toward ~500 and stays there; the no-clip ablation rises then
# destabilizes. (Exact numbers vary by hardware/seed; our small run, not the paper's.)

## Visualize the data & results

_Does PPO's clipped objective make the episode return rise to the solved score on CartPole, and does removing the clip (raw r_t*A_t, same everything else) destabilize it? We train both for the same updates and plot the average return per update._

In [ ]:
# Sketch of how the two curves above are produced (full loop in the CODE cell).
# Train clipped PPO and the no-clip ablation on CartPole-v1 for the same number of
# updates with identical net / GAE / lr / epochs / seed, recording avg return per update.
#
#   clip_hist   = train(use_clip=True)    # green: climbs to ~500 and stays (SOLVED)
#   noclip_hist = train(use_clip=False)   # red:   rises then oscillates / crashes
#
# The points plotted are the per-update average return (last 20 episodes).
# Clipped -> smooth monotone climb past the 475 "solved" line.
# No-clip -> spikes up and crashes down repeatedly: removing Eq. 7's clip lets a
# single reused-batch update push a good action's ratio huge and break the policy.
# (Numbers are from our own run; the paper reports MuJoCo Table-1 score 0.82 at eps=0.2
#  and Atari Table-2 wins, not these CartPole values.)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The worked clip. With $\epsilon = 0.2$, the old policy gave an action probability $0.5$, the
            new policy raised it to $0.8$, and the advantage was $\hat{A}_t = +2$. Compute the unclipped term,
            the clipped term, the value of $L^{CLIP}$ for this sample, and its gradient with respect to $r_t$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Form the ratio: $r_t = 0.8 / 0.5 = 1.6$. — _The ratio is new-over-old probability; here the new policy made the action $1.6\times$ as likely._
- Unclipped term: $r_t \hat{A}_t = 1.6 \times 2 = 3.2$. — _That is $L^{CPI}$ for this sample &mdash; what we'd get with no clip._
- Clip then multiply: $\text{clip}(1.6, 0.8, 1.2) = 1.2$, so $1.2 \times 2 = 2.4$. — _$1.6$ exceeds $1+\epsilon = 1.2$, so the ratio is clamped to $1.2$._
- Take the min: $\min(3.2, 2.4) = 2.4$. — _For $\hat{A}_t \gt 0$ above the clip, the clipped (flat) term is smaller, so the $\min$ selects it._

**Answer:** $L^{CLIP} = 2.4$ for this sample, and because the clipped (flat) term is selected, the
                 gradient with respect to $r_t$ is $0$ &mdash; PPO gives no incentive to push this already-favored
                 good action's probability higher in this update. The notebook recomputes
                 $\min(1.6 \times 2,\, 1.2 \times 2) = 2.4$.

</details>

**Problem 2.** The ablation. Your PPO agent solves CartPole (return climbs to ~500). Replace the clipped
            loss with the raw surrogate $-\,(r_t \hat{A}_t).\text{mean}()$ &mdash; keeping everything else
            (network, GAE, epochs, data) identical &mdash; and retrain. What happens to the return curve, and
            what does that demonstrate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Change only the policy loss: delete the clip + min, use -(ratio * adv).mean(); keep depth, optimizer, epochs, and seed fixed. — _An honest ablation changes exactly one thing &mdash; the clip &mdash; so any difference is attributable to it._
- Retrain and watch the return: the unclipped run rises then collapses or oscillates wildly, while the clipped run climbed and stayed near 500. — _Without the clip, reusing each batch for several epochs lets a single update push a good-looking action's ratio huge, overfitting the batch and breaking the policy._
- Conclude the clip, not the network or GAE, is what makes the multi-epoch reuse safe. — _Both runs share architecture and data; only the clipped one is stable, isolating the clip as the cause._

**Answer:** The unclipped agent destabilizes &mdash; its return spikes and crashes as the policy
                 over-updates on reused batches &mdash; while the clipped PPO agent climbs to ~500 and holds.
                 Since the only difference is Eq. 7's clip, this isolates the clipped surrogate as the source of
                 stability: it is the ingredient that makes several-epoch reuse of on-policy data safe. The
                 CODEVIZ panel shows this contrast.

</details>

**Problem 3.** Now suppose the same action ($r_t = 1.6$) had a negative advantage $\hat{A}_t = -2$ &mdash;
            an over-shot bad action. With $\epsilon = 0.2$, what does $L^{CLIP}$ equal for this sample, and why
            does the $\min$ keep a useful gradient here?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Unclipped: $r_t \hat{A}_t = 1.6 \times (-2) = -3.2$. — _A bad action made more likely scores very negatively under the raw surrogate._
- Clipped: $\text{clip}(1.6, 0.8, 1.2) \times (-2) = 1.2 \times (-2) = -2.4$. — _The ratio clamps to $1.2$; multiplied by the negative advantage gives $-2.4$._
- Take the min: $\min(-3.2, -2.4) = -3.2$. — _For $\hat{A}_t \lt 0$, the unclipped term is more negative, so the $\min$ selects it &mdash; restoring a gradient._

**Answer:** $L^{CLIP} = -3.2$ &mdash; the unclipped term wins. Because this bad action's ratio
                 already over-shot to $1.6$, the $\min$ reactivates the unclipped surrogate, whose gradient
                 pulls the action's probability back down. The clip-only objective would have frozen at $-2.4$
                 (flat, no gradient) and stranded the mistake; the $\min$ is what guarantees a past overshoot
                 stays correctable.

</details>